In [6]:
import json
import numpy as np
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import random
import pickle
from sklearn.metrics import precision_recall_fscore_support
from keras.models import load_model

In [7]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [8]:
train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')

In [9]:
data_for_dataframe = []
evidence_keys = list(evidence_map.keys())  # List of all evidence IDs

for claim_id, claim_details in train_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

    # Add positive examples
    for eid in claim_evidences:
        evidence_text = evidence_map.get(eid, "NULL")  
        if evidence_text != "NULL":
            data_for_dataframe.append({
                'claim': claim_text,
                'evidence': evidence_text,
                'label': 1  # Label as relevant
            })

    # Add negative examples
    num_neg_samples = min(len(claim_evidences), len(evidence_keys) - len(claim_evidences))  # Limit the number of negative samples
    negative_samples = random.sample([k for k in evidence_keys if k not in claim_evidences], num_neg_samples)
    for eid in negative_samples:
        evidence_text = evidence_map[eid]
        data_for_dataframe.append({
            'claim': claim_text,
            'evidence': evidence_text,
            'label': 0  # Label as not relevant
        })

train_df = pd.DataFrame(data_for_dataframe)

In [15]:
def text2seq(train_text, tokenizer_name):
	tokenizer = Tokenizer(oov_token='<UNK>')
	tokenizer.fit_on_texts(train_text)
	input_text_index = tokenizer.word_index # return dictionary of wordss {'the':1, 'earth':2, 'is':3}

	with open(tokenizer_name+'.pickle', 'wb') as handle:
		pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

	max_length = max([len(s.split()) for s in train_text])
	print ("max length:", max_length)

	train_sequence = tokenizer.texts_to_sequences(train_text)
	return (train_sequence, input_text_index, max_length)

def to_padding(train_df):
	# Initialize and fit the tokenizer on claim and evidence separately
	x_claims_seq, x_claims_word_index, max_claims_length = text2seq(train_df["claim"].tolist(), "tokenizer_claims_full")
	x_sents_seq, x_sents_word_index, max_sents_length = text2seq(train_df["evidence"].tolist(), "tokenizer_evidence_full")

	x_claims_data = pad_sequences(x_claims_seq, maxlen=max_claims_length)  #returns array of data
	x_sents_data = pad_sequences(x_sents_seq, maxlen=max_sents_length)
	x_labels = train_df['label'].values

	return (x_claims_data, x_sents_data, x_labels, x_claims_word_index,  x_sents_word_index)

def create_embedding_matrix(vocab_size, word_vectors, word_index, embedding_dim):
	embedding_matrix = np.zeros((vocab_size, embedding_dim))
	for word, i in word_index.items():
		if word in word_vectors and i < vocab_size:
			embedding_vector = word_vectors[word]
			if embedding_vector is not None:
				embedding_matrix[i] = embedding_vector
	return embedding_matrix, embedding_dim

In [22]:
x_claim, x_sents, x_labels, x_claims_word_index,  x_sents_word_index = to_padding(train_df)
loss = 'binary_crossentropy'

print ("x claim word index ", len(x_claims_word_index))
print ("x sent word index ", len(x_sents_word_index))

# vocab_size_claims = len(x_claims_word_index) + 1
# vocab_size_evidences = len(x_sents_word_index) + 1
vocab_size_claims = 3000
vocab_size_evidences = 16000

max length: 35
max length: 180
x claim word index  2749
x sent word index  15250


In [23]:
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')

In [24]:
embedding_dim = 300  # dimension of word2vec vectors
(embed_matrix_claim, embed_dim_claim) = create_embedding_matrix(vocab_size_claims, word_vectors, x_claims_word_index, embedding_dim)
(embed_matrix_evidence, embed_dim_evidence) = create_embedding_matrix(vocab_size_evidences, word_vectors, x_sents_word_index, embedding_dim)

print ("embed_matrix_claim shape ", embed_matrix_claim.shape)
print ("embed_matrix_evidence shape ", embed_matrix_evidence.shape)

embed_matrix_claim shape  (3000, 300)
embed_matrix_evidence shape  (16000, 300)


In [25]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout, concatenate, BatchNormalization

# Define the model
def create_model(vocab_size_claims, vocab_size_evidences, maxlen_claims, maxlen_evidences, 
								embed_dim_claim, embed_dim_evidence):
		
	claims_input = Input(shape=(None,), dtype='int32', name='claims')
	embed_claims = Embedding(vocab_size_claims, embed_dim_claim)(claims_input)
	encoded_claims = LSTM(256, return_sequences=True)(embed_claims)
	encoded_claims = LSTM(16)(encoded_claims)
	encoded_claims = BatchNormalization()(encoded_claims)
		
	evidences_input = Input(shape=(None,), dtype='int32', name='evidences')
	embed_evidences = Embedding(vocab_size_evidences, embed_dim_evidence)(evidences_input)
	encoded_evidences = LSTM(256, return_sequences=True)(embed_evidences)
	encoded_evidences= LSTM(64)(encoded_evidences)
	encoded_evidences = BatchNormalization()(encoded_evidences)
		
	concatenate_layers = concatenate([encoded_claims, encoded_evidences], axis=-1)
	concatenate_layers = Dropout(0.5)(concatenate_layers)
	concatenate_layers = Dense(64, activation='relu')(concatenate_layers)
	pred_label = Dense(1, activation='sigmoid')(concatenate_layers)

	model = Model(inputs=[claims_input, evidences_input], outputs=pred_label)
	model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

	return model


max_claims_length = 35
max_sents_length = 180

model = create_model(vocab_size_claims, vocab_size_evidences, max_claims_length, max_sents_length, 
								embed_dim_claim, embed_dim_evidence)

model.layers[2].set_weights([embed_matrix_claim])
model.layers[2].trainable = False
model.layers[3].set_weights([embed_matrix_evidence])
model.layers[3].trainable = False

print(model.summary())


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 claims (InputLayer)         [(None, None)]               0         []                            
                                                                                                  
 evidences (InputLayer)      [(None, None)]               0         []                            
                                                                                                  
 embedding_2 (Embedding)     (None, None, 300)            900000    ['claims[0][0]']              
                                                                                                  
 embedding_3 (Embedding)     (None, None, 300)            4800000   ['evidences[0][0]']           
                                                                                            

In [26]:
from keras.callbacks import EarlyStopping, ModelCheckpoint, Callback, CSVLogger
early_stopping = EarlyStopping(monitor='val_loss', patience=2)

csv_logger = CSVLogger('lstm_training_tf.log')
model_path = 'full_lstm_evidence_retrieval.hdf5'
history = model.fit({'claims': x_claim, 'evidences': x_sents}, x_labels, 
						epochs=60, batch_size=64, validation_split=0.12, callbacks=[early_stopping, csv_logger,
						ModelCheckpoint(filepath=model_path, monitor='val_loss', save_best_only=True)])	

Epoch 1/60
114/114 [==============================] - 110s 928ms/step - loss: 0.2205 - accuracy: 0.9214 - val_loss: 0.2250 - val_accuracy: 0.9343
Epoch 2/60


/Users/chenluyao/opt/anaconda3/envs/nlp/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


114/114 [==============================] - 94s 828ms/step - loss: 0.1456 - accuracy: 0.9506 - val_loss: 0.1743 - val_accuracy: 0.9354
Epoch 3/60
114/114 [==============================] - 129s 1s/step - loss: 0.1045 - accuracy: 0.9653 - val_loss: 0.1772 - val_accuracy: 0.9364
Epoch 4/60
114/114 [==============================] - 111s 971ms/step - loss: 0.0722 - accuracy: 0.9738 - val_loss: 0.1768 - val_accuracy: 0.9455


In [27]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 

,claim_id,claim,evidence
0,claim-752,south australia expens electr world,"[evidence-67732, evidence-572512]"
1,claim-375,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2..."
2,claim-1266,mean world 1c warmer time,"[evidence-889933, evidence-694262]"
3,claim-871,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28..."
4,claim-2164,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947..."
...,...,...,...
149,claim-2400,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85..."
150,claim-204,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20..."
151,claim-1426,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1..."
152,claim-698,recent studi led lawrenc livermor nation labor...,[evidence-660755]


In [29]:
with open('tokenizer_claims_full.pickle', 'rb') as handle:
	claims_tokenizer = pickle.load(handle)

with open('tokenizer_evidence_full.pickle', 'rb') as handle:
	sents_tokenizer = pickle.load(handle)

max_claims_length = 35
max_sents_length = 180

model = load_model("full_lstm_evidence_retrieval.hdf5") # OR hdf5 file

In [30]:
test_claims = claims_tokenizer.texts_to_sequences(dev_claims_df["claim"])
test_sents = sents_tokenizer.texts_to_sequences(evidence_map.values())

test_claims = pad_sequences(test_claims, maxlen=max_claims_length)
test_sents = pad_sequences(test_sents, maxlen=max_sents_length)
print ("test claims ", test_claims.shape)
print ("test sents ", test_sents.shape)

test claims  (154, 35)
test sents  (1208827, 180)


In [32]:
for i in range(test_claims.shape[0]):
    claim_row = test_claims[i]  # Retrieve one row of test_claims

    # Replicate this row to match the number of rows in test_sents
    replicated_claims = np.tile(claim_row, (test_sents.shape[0], 1))

    # Now, create the dictionary to feed into the model
    input_dict = {
        'claims': replicated_claims,
        'evidences': test_sents
    }

    # Predict using the model
    y_pred = model.predict(input_dict, batch_size=128)
    y_pred = np.asarray(y_pred).round()
    print("Y_PREDICT: ", y_pred)

    # Assuming y_labels is properly aligned with these predictions
    # Calculate precision, recall, and F1-score
    # scores = precision_recall_fscore_support(y_labels, y_pred, average='binary')
    # print(f"Score of LSTM for claim {i+1}: Precision={scores[0]}, Recall={scores[1]}, F1-Score={scores[2]}")

 986/9444 [==>...........................] - ETA: 21:11:28

KeyboardInterrupt: 